In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
df = pd.read_csv("spotify_tracks.csv")

df['track_name'] = df['track_name'].str.lower()
df = df.drop_duplicates(subset=['track_name', 'artist_name'])
df = df.dropna()

In [5]:
df = df.reset_index(drop=True)

In [6]:
features = [
    'acousticness','danceability','energy','instrumentalness',
    'liveness','loudness','speechiness','tempo','valence'
]

In [7]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[features])

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_cosine(song_name):
    
    song_name = song_name.lower()
    
    matches = df[df['track_name'] == song_name]
    
    if matches.empty:
        return "Song not found"
    
    idx = matches.sort_values('popularity', ascending=False).index[0]
    
    song_vector = scaled_data[idx].reshape(1, -1)
    
    similarity = cosine_similarity(song_vector, scaled_data)[0]
    
    # popularity boost
    pop = df['popularity'].values
    similarity = (0.9 * similarity) + (0.1 * (pop / pop.max()))
    
    # artist penalty
    artist = df.iloc[idx]['artist_name']
    artist_match = (df['artist_name'] == artist).astype(int)
    similarity = similarity - (0.2 * artist_match)
    
    # ✅ balanced language control
    lang = df.iloc[idx]['language']
    lang_match = (df['language'] == lang).astype(int)
    similarity = similarity + (0.04 * lang_match)
    similarity = similarity - (0.04 * (1 - lang_match))
    
    # remove itself
    similarity[idx] = -1
    
    sorted_idx = similarity.argsort()[::-1]
    
    results = []
    seen_artists = set()
    seen_names = set()
    
    for i in sorted_idx:
        
        track = df.iloc[i]['track_name']
        artist_i = df.iloc[i]['artist_name']
        
        if similarity[i] < 0.3:
            continue
        
        if artist_i in seen_artists:
            continue
        
        if track in seen_names:
            continue
        
        results.append(f"{track} - {artist_i}")
        seen_artists.add(artist_i)
        seen_names.add(track)
        
        if len(results) == 10:
            break
    
    # ✅ fallback safety
    if len(results) < 10:
        for i in sorted_idx:
            track = df.iloc[i]['track_name']
            artist_i = df.iloc[i]['artist_name']
            
            if track not in seen_names:
                results.append(f"{track} - {artist_i}")
            
            if len(results) == 10:
                break
    
    return results

In [9]:
recommend_cosine("raakh")

['dil ke paas (indian version) [from "wajah tum ho"] - Arijit Singh, Tulsi Kumar',
 'rab ka shukrana - reprise - Pritam, Anupam Amod',
 'dil ke paas (indian version) - Arijit Singh, Tulsi Kumar, Abhijit Vaghani, Kalyanji-Anandji',
 'kyun dil mera - from "paharganj" - Mohit Chauhan',
 'the 1 - Taylor Swift',
 'mere dholna revisited - Pritam, Shreya Ghoshal',
 'mazhai kuruvi - A.R. Rahman',
 'taarefon se - Amit Trivedi, Arijit Singh',
 'raabta - Pritam, Arijit Singh',
 'naan nee - Santhosh Narayanan, Shakthisree Gopalan, Dhee']

In [10]:
recommend_cosine("in pieces")

['bye bye bye - from deadpool and wolverine soundtrack - *NSYNC',
 'rock me - One Direction',
 "i know places (taylor's version) - Taylor Swift",
 "i'll be waiting - Adele",
 '(entre paréntesis) - Shakira, Grupo Frontera',
 'do ya thang - Rihanna',
 'big shot - Billy Joel',
 'pray - Justin Bieber',
 'shooting star - Yuvan Shankar Raja, Adithya RK',
 'mama kin - Aerosmith']

In [11]:
recommend_cosine("calling")

['silhouettes - original radio edit - Avicii',
 'magic - One Direction',
 'pressure - Billy Joel',
 'bye bye bye - from deadpool and wolverine soundtrack - *NSYNC',
 'a song for the drunk and broken hearted - Passenger',
 'till death do us part - Madonna',
 'shooting star - Yuvan Shankar Raja, Adithya RK',
 "can't remember to forget you (feat. rihanna) - Shakira, Rihanna",
 'kadal pole (from "kummattikali") - Yuvan Shankar Raja, Sumesh Parameshwaran, Santhosh Varma',
 'in pieces - Linkin Park']

In [12]:
df.sample(5)

,track_id,track_name,artist_name,year,popularity,artwork_url,album_name,acousticness,danceability,duration_ms,...,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,track_url,language
11156,6Sa45NfAeYfkHQPqgV1D1W,maahi ve,"Shankar-Ehsaan-Loy, Sadhana Sargam, Sujata Bha...",2017,11,https://i.scdn.co/image/ab67616d0000b27306e83d...,Shah Rukh Khan - Raja of Bollywood,0.029700,0.815,366853.0,...,7.0,0.330,-6.117,0.0,0.0771,99.989,4.0,0.562,https://open.spotify.com/track/6Sa45NfAeYfkHQP...,Unknown
37425,6of1dXbW31YXjbrz9HzA4X,ugomeki,THE MADNA,2022,10,https://i.scdn.co/image/ab67616d0000b2730ade3a...,Ugly heaven,0.000008,0.520,165000.0,...,9.0,0.219,-3.542,1.0,0.0648,140.025,4.0,0.659,https://open.spotify.com/track/6of1dXbW31YXjbr...,English
9435,2vtHr8V7RZhNW92lVXBURA,rammaa seetha raadhamma,"D. Imman, Hemachandra Vedala, Vandana Srinivasan",2019,9,https://i.scdn.co/image/ab67616d0000b27328c2e4...,Seemaraja (Original Motion Picture Soundtrack),0.298000,0.390,264907.0,...,7.0,0.268,-3.475,1.0,0.0798,90.017,3.0,0.760,https://open.spotify.com/track/2vtHr8V7RZhNW92...,Tamil
821,3J1sBhSdSuKWBmxmGz82M4,"kannile (from ""andhagan"")","Santhosh Narayanan, Adithya RK",2023,15,https://i.scdn.co/image/ab67616d0000b273e34155...,"Kannile (From ""Andhagan"")",0.961000,0.573,182972.0,...,10.0,0.102,-11.100,0.0,0.0362,119.913,4.0,0.172,https://open.spotify.com/track/3J1sBhSdSuKWBmx...,Unknown
18995,0leIXrxhkEBGoR8O8PZQEg,outro. incompletion,SEVENTEEN,2017,41,https://i.scdn.co/image/ab67616d0000b27346a157...,"SEVENTEEN 2ND ALBUM 'TEEN, AGE' (2)",0.000325,0.592,73760.0,...,11.0,0.274,-5.520,1.0,0.0341,100.081,4.0,0.637,https://open.spotify.com/track/0leIXrxhkEBGoR8...,Unknown


In [13]:
import pickle
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)